# Parte 1 — Datamart analítico y ETL

**Asignatura:** Inteligencia de Negocios — Proyecto Grupal  
**Rol:** Arquitecto de BI  
**Objetivo:** Validar el datamart dimensional (esquema estrella), documentar el proceso ETL y dejar las tablas listas para Power BI.

## Contexto

Los datos sintéticos fueron generados por el equipo (scripts en `scripts/`). La versión oficial para análisis está en `data/processed/`. Este notebook:

1. Carga las tablas dimensionales y de hechos.
2. Reporta métricas de calidad **antes** del procesamiento.
3. Aplica estandarización y validaciones de integridad referencial.
4. Verifica métricas derivadas (`importe_venta`, `margen`).
5. Reporta calidad **después** del ETL.
6. Construye el diccionario de datos y confirma el esquema estrella.

**Grano de la tabla de hechos:** una fila = una línea de venta (`id_venta` + `numero_linea`).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Rutas del proyecto (funciona desde notebooks/ o desde la raíz)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

SEP = ";"
ENC = "utf-8"
RANDOM_STATE = 42

TABLAS = {
    "dim_cliente": "dim_cliente.csv",
    "dim_producto": "dim_producto.csv",
    "dim_tienda": "dim_tienda.csv",
    "dim_promocion": "dim_promocion.csv",
    "dim_tiempo": "dim_tiempo.csv",
    "fact_ventas": "fact_ventas.csv",
}

print("Raíz del proyecto: .")
print("Carpeta de datos: data/processed")

In [ ]:
def cargar_tabla(nombre: str) -> pd.DataFrame:
    path = DATA_PROCESSED / TABLAS[nombre]
    if not path.exists():
        raise FileNotFoundError(f"No se encontró {path}")
    return pd.read_csv(path, sep=SEP, encoding=ENC)


def reporte_calidad(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    filas = len(df)
    nulos = df.isna().sum()
    pct_nulos = (nulos / filas * 100).round(2) if filas else 0
    resumen = pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": nulos.values,
        "pct_nulos": pct_nulos.values if hasattr(pct_nulos, "values") else pct_nulos,
        "unicos": [df[c].nunique() for c in df.columns],
    })
    print(f"\n=== Calidad inicial: {nombre} ({filas:,} filas) ===")
    display(resumen)
    return resumen


def contar_duplicados_grano(df: pd.DataFrame, columnas: list[str]) -> int:
    return int(df.duplicated(subset=columnas).sum())


def contar_huérfanas(fact_col, dim_col) -> int:
    valores_fact = set(fact_col.dropna())
    valores_dim = set(dim_col.dropna())
    return len(valores_fact - valores_dim)

In [ ]:
# 1) Carga de tablas desde data/processed
raw = {nombre: cargar_tabla(nombre) for nombre in TABLAS}

for nombre, df in raw.items():
    print(f"{nombre}: {len(df):,} filas x {df.shape[1]} columnas")

calidad_antes = {}
for nombre, df in raw.items():
    calidad_antes[nombre] = reporte_calidad(df, nombre)

In [ ]:
# 2) Copia de trabajo para ETL
etl = {k: v.copy() for k, v in raw.items()}
fact = etl["fact_ventas"]

# Estandarizar fechas
fact["fecha"] = pd.to_datetime(fact["fecha"], errors="coerce")
etl["dim_tiempo"]["fecha"] = pd.to_datetime(etl["dim_tiempo"]["fecha"], errors="coerce")
etl["dim_cliente"]["fecha_alta"] = pd.to_datetime(etl["dim_cliente"]["fecha_alta"], errors="coerce")
etl["dim_cliente"]["fecha_nacimiento"] = pd.to_datetime(
    etl["dim_cliente"]["fecha_nacimiento"], errors="coerce"
)
etl["dim_promocion"]["fecha_inicio"] = pd.to_datetime(
    etl["dim_promocion"]["fecha_inicio"], errors="coerce", dayfirst=True
)
etl["dim_promocion"]["fecha_fin"] = pd.to_datetime(
    etl["dim_promocion"]["fecha_fin"], errors="coerce", dayfirst=True
)

# Normalizar texto en categorías
etl["dim_producto"]["categoria"] = (
    etl["dim_producto"]["categoria"].astype(str).str.strip().str.title()
)

# Tratar nulos en descuento (regla de negocio: sin descuento = 0)
fact["descuento_pct"] = pd.to_numeric(fact["descuento_pct"], errors="coerce").fillna(0)

duplicados_antes = contar_duplicados_grano(fact, ["id_venta", "numero_linea"])
print(f"Duplicados por grano (id_venta + numero_linea): {duplicados_antes}")

# Eliminar duplicados por clave de negocio
fact = fact.drop_duplicates(subset=["id_venta", "numero_linea"], keep="first")
etl["fact_ventas"] = fact

print(f"Filas después de deduplicación: {len(fact):,}")

In [ ]:
# 3) Integridad referencial
fk_checks = [
    ("id_cliente", etl["dim_cliente"], "id_cliente", "fact_ventas → dim_cliente"),
    ("id_producto", etl["dim_producto"], "id_producto", "fact_ventas → dim_producto"),
    ("id_tienda", etl["dim_tienda"], "id_tienda", "fact_ventas → dim_tienda"),
    ("id_promocion", etl["dim_promocion"], "id_promocion", "fact_ventas → dim_promocion"),
    ("fecha", etl["dim_tiempo"], "fecha", "fact_ventas → dim_tiempo"),
]

fk_resultados = []
for col_fact, dim, col_dim, etiqueta in fk_checks:
    huerfanas = contar_huérfanas(fact[col_fact], dim[col_dim])
    fk_resultados.append({"relacion": etiqueta, "claves_huerfanas": huerfanas})
    status = "OK" if huerfanas == 0 else "ERROR"
    print(f"{status} {etiqueta}: {huerfanas} huérfanas")

display(pd.DataFrame(fk_resultados))

if any(r["claves_huerfanas"] > 0 for r in fk_resultados):
    raise ValueError("Hay claves foráneas huérfanas. Revise el ETL.")

In [ ]:
# 4) Validación de métricas derivadas
fact = etl["fact_ventas"]

importe_calc = (
    fact["cantidad"]
    * fact["precio_unitario_final"].astype(float)
).round(2)
margen_calc = (fact["importe_venta"] - fact["costo_total"]).round(2)

diff_importe = (fact["importe_venta"] - importe_calc).abs()
diff_margen = (fact["margen"] - margen_calc).abs()

print("Diferencia máxima importe_venta calculado vs almacenado:", diff_importe.max())
print("Diferencia máxima margen calculado vs almacenado:", diff_margen.max())
print("Filas con cantidad <= 0:", (fact["cantidad"] <= 0).sum())
print("Filas con importe_venta < 0:", (fact["importe_venta"] < 0).sum())

assert (fact["cantidad"] > 0).all(), "Hay cantidades inválidas"
assert diff_importe.max() < 0.05, "importe_venta no cuadra con cantidad * precio final"
assert diff_margen.max() < 0.05, "margen no cuadra con importe - costo"
print("\nOK métricas derivadas consistentes.")

In [ ]:
# 5) Reporte de calidad DESPUÉS del ETL
calidad_despues = {}
for nombre, df in etl.items():
    calidad_despues[nombre] = reporte_calidad(df, f"{nombre} (post-ETL)")

comparativo = []
for key in calidad_antes:
    comparativo.append({
        "tabla": key,
        "nulos_antes": int(calidad_antes[key]["nulos"].sum()),
        "nulos_despues": int(calidad_despues[key]["nulos"].sum()),
        "filas": len(etl[key]),
    })

display(pd.DataFrame(comparativo))

## Esquema estrella (modelo dimensional)

```mermaid
erDiagram
    FACT_VENTAS ||--o{ DIM_CLIENTE : id_cliente
    FACT_VENTAS ||--o{ DIM_PRODUCTO : id_producto
    FACT_VENTAS ||--o{ DIM_TIENDA : id_tienda
    FACT_VENTAS ||--o{ DIM_PROMOCION : id_promocion
    FACT_VENTAS ||--o{ DIM_TIEMPO : fecha
```

| Elemento | Descripción |
| --- | --- |
| **Grano** | Línea de venta (`id_venta` + `numero_linea`) |
| **Hechos** | cantidad, precios, descuento, importe_venta, costo_total, margen |
| **Dimensiones** | Cliente, Producto, Tienda, Promoción, Tiempo |

In [ ]:
# 6) Diccionario de datos
DICCIONARIO = [
    # fact_ventas
    ("fact_ventas", "id_venta", "int", "PK lógica del ticket", "No"),
    ("fact_ventas", "numero_linea", "int", "Número de línea dentro del ticket", "No"),
    ("fact_ventas", "fecha", "date", "Fecha de la venta (FK → dim_tiempo)", "Sí"),
    ("fact_ventas", "id_cliente", "int", "Cliente (FK → dim_cliente)", "Sí"),
    ("fact_ventas", "id_tienda", "int", "Tienda/canal (FK → dim_tienda)", "Sí"),
    ("fact_ventas", "id_producto", "int", "Producto (FK → dim_producto)", "Sí"),
    ("fact_ventas", "id_promocion", "int", "Promoción aplicada (FK → dim_promocion)", "Sí"),
    ("fact_ventas", "cantidad", "int", "Unidades vendidas", "No"),
    ("fact_ventas", "precio_unitario_lista", "float", "Precio de lista unitario", "No"),
    ("fact_ventas", "descuento_pct", "float", "Descuento aplicado (0-1)", "No"),
    ("fact_ventas", "precio_unitario_final", "float", "Precio unitario neto", "No"),
    ("fact_ventas", "importe_venta", "float", "Métrica: ingreso de la línea", "No"),
    ("fact_ventas", "costo_total", "float", "Costo total de la línea", "No"),
    ("fact_ventas", "margen", "float", "Métrica: importe_venta - costo_total", "No"),
    # dim_cliente
    ("dim_cliente", "id_cliente", "int", "PK del cliente", "Sí"),
    ("dim_cliente", "nombre", "str", "Nombre del cliente", "No"),
    ("dim_cliente", "sexo", "str", "Sexo (M/F)", "No"),
    ("dim_cliente", "fecha_nacimiento", "date", "Fecha de nacimiento", "No"),
    ("dim_cliente", "distrito", "str", "Distrito de residencia", "No"),
    ("dim_cliente", "region", "str", "Región", "No"),
    ("dim_cliente", "fecha_alta", "date", "Primera compra / alta en programa", "No"),
    ("dim_cliente", "segmento_programa", "str", "Segmento fidelización (Bronce, Plata, Oro...)", "No"),
    # dim_producto
    ("dim_producto", "id_producto", "int", "PK del producto", "Sí"),
    ("dim_producto", "nombre", "str", "Nombre del producto", "No"),
    ("dim_producto", "categoria", "str", "Categoría comercial", "No"),
    ("dim_producto", "subcategoria", "str", "Subcategoría", "No"),
    ("dim_producto", "marca", "str", "Marca", "No"),
    ("dim_producto", "precio_lista", "float", "Precio de lista", "No"),
    ("dim_producto", "costo_unitario_promedio", "float", "Costo unitario promedio", "No"),
    ("dim_producto", "producto_estrella", "bool", "Indicador producto estrella (Pareto)", "No"),
    # dim_tienda
    ("dim_tienda", "id_tienda", "int", "PK de tienda/canal", "Sí"),
    ("dim_tienda", "nombre", "str", "Nombre de la tienda", "No"),
    ("dim_tienda", "canal", "str", "Físico u Online", "No"),
    ("dim_tienda", "region", "str", "Región de operación", "No"),
    ("dim_tienda", "ciudad", "str", "Ciudad", "No"),
    # dim_promocion
    ("dim_promocion", "id_promocion", "int", "PK de promoción", "Sí"),
    ("dim_promocion", "nombre", "str", "Nombre de la campaña", "No"),
    ("dim_promocion", "tipo", "str", "Tipo de promoción", "No"),
    ("dim_promocion", "descuento_pct", "float", "Porcentaje de descuento", "No"),
    ("dim_promocion", "fecha_inicio", "date", "Inicio de vigencia", "No"),
    ("dim_promocion", "fecha_fin", "date", "Fin de vigencia", "No"),
    # dim_tiempo
    ("dim_tiempo", "fecha", "date", "PK del calendario", "Sí"),
    ("dim_tiempo", "dia", "int", "Día del mes", "No"),
    ("dim_tiempo", "mes", "int", "Mes", "No"),
    ("dim_tiempo", "trimestre", "int", "Trimestre", "No"),
    ("dim_tiempo", "anio", "int", "Año", "No"),
    ("dim_tiempo", "dia_semana", "str", "Nombre del día", "No"),
    ("dim_tiempo", "es_feriado", "bool", "Indicador de feriado", "No"),
]

df_diccionario = pd.DataFrame(
    DICCIONARIO,
    columns=["tabla", "campo", "tipo", "descripcion", "es_clave_fk"],
)
display(df_diccionario.head(15))
print(f"\nTotal campos documentados: {len(df_diccionario)}")

ruta_diccionario = DOCS_DIR / "diccionario_datos.md"
with ruta_diccionario.open("w", encoding="utf-8") as f:
    f.write("# Diccionario de datos — Datamart retail\n\n")
    f.write("| Tabla | Campo | Tipo | Descripción | Clave/FK |\n")
    f.write("| --- | --- | --- | --- | --- |\n")
    for row in DICCIONARIO:
        f.write(f"| {row[0]} | {row[1]} | {row[2]} | {row[3]} | {row[4]} |\n")

print("Diccionario exportado a: docs/diccionario_datos.md")

In [ ]:
# 7) Exportar tablas validadas (confirmación para Power BI)
OUTPUT = DATA_PROCESSED
for nombre, df in etl.items():
    out = OUTPUT / TABLAS[nombre]
    df_export = df.copy()
    if "fecha" in df_export.columns and pd.api.types.is_datetime64_any_dtype(df_export["fecha"]):
        df_export["fecha"] = df_export["fecha"].dt.strftime("%Y-%m-%d")
    df_export.to_csv(out, sep=SEP, index=False, encoding=ENC)
    print(f"Exportado: {out.name} ({len(df_export):,} filas)")

print("\n=== RESUMEN FINAL DEL DATAMART ===")
print(f"Tickets: {etl['fact_ventas']['id_venta'].nunique():,}")
print(f"Líneas: {len(etl['fact_ventas']):,}")
print(f"Ventas totales: S/ {etl['fact_ventas']['importe_venta'].sum():,.2f}")
print(f"Margen total: S/ {etl['fact_ventas']['margen'].sum():,.2f}")
print(f"Periodo: {etl['fact_ventas']['fecha'].min()} a {etl['fact_ventas']['fecha'].max()}")
print("\nDatamart listo para importar en Power BI desde data/processed/.")

## Conclusiones — Parte 1

- El datamart cumple el **esquema estrella** con `fact_ventas` en el centro.
- La calidad del dataset es **consistente**: sin nulos críticos, sin duplicados de grano y sin claves huérfanas.
- Las métricas `importe_venta` y `margen` están correctamente calculadas.
- El diccionario de datos fue exportado a `docs/diccionario_datos.md`.
- **Siguiente paso:** importar las tablas en Power BI (`powerbi/README.md`) y construir el tablero (Parte 2).